# 09. PyIceberg

PyIceberg は、JVM（Java）を使わない純粋な Python 製の Iceberg クライアントです。
Spark や Trino のような重いエンジンがなくても、Python だけで同じテーブルを読み書きできます。

このノートブックでは Spark を使いません。

- テーブル: `handson.py_trips`
- 事前に `make data` でデータを取得しておく

## 1. カタログに接続する

接続設定（Polaris の URL、認証情報）は環境変数 `PYICEBERG_CATALOG__LAKEHOUSE__*` で渡してあるので、名前を指定するだけで接続できます。
他のエンジンで作ったテーブルも見えます。

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("lakehouse")
print(catalog.list_namespaces())
for identifier in catalog.list_tables("handson"):
    print(".".join(identifier))

## 2. テーブルを作って書き込む

pyarrow で読んだ Parquet の一部（2025-01-01 の乗車）から、テーブルを作って追記します。
スキーマは pyarrow のスキーマから決まります。

In [ ]:
from datetime import datetime
import pyarrow.compute as pc
import pyarrow.parquet as pq

columns = ["VendorID", "tpep_pickup_datetime", "trip_distance", "payment_type", "total_amount"]
jan = pq.read_table("/workspace/data/yellow_tripdata_2025-01.parquet", columns=columns)

def one_day(day):
    """指定した日の乗車だけを取り出す"""
    start = datetime(2025, 1, day)
    end = datetime(2025, 1, day + 1)
    ts = jan["tpep_pickup_datetime"]
    return jan.filter(pc.and_(pc.greater_equal(ts, start), pc.less(ts, end)))

if catalog.table_exists("handson.py_trips"):
    catalog.purge_table("handson.py_trips")

day1 = one_day(1)
table = catalog.create_table("handson.py_trips", schema=day1.schema)
table.append(day1)
print(table.schema())
print("行数:", day1.num_rows)

もう 1 日分を追記します。書き込みのたびにスナップショットが増えるのは、他のエンジンと同じです。

In [ ]:
table.append(one_day(2))
for snapshot in table.snapshots():
    print(snapshot.snapshot_id, snapshot.summary.operation, snapshot.summary["added-records"])

## 3. 条件を付けて読む

`scan` に行の条件（`row_filter`）と列（`selected_fields`）を渡すと、
Iceberg のメタデータ（ファイルごとの統計）で関係ないファイルを読み飛ばしたうえで、必要な列だけを読みます。

In [ ]:
df = table.scan(
    row_filter="trip_distance >= 30",
    selected_fields=("tpep_pickup_datetime", "trip_distance", "total_amount"),
).to_pandas()
print(len(df), "行")
df.sort_values("trip_distance", ascending=False).head()

走行距離が数万マイルの行は、実データに含まれる異常値です。分析するときは、こうした値を除く条件も必要になります。

## 4. DuckDB で SQL を書く

スキャン結果を DuckDB に渡すと、Python の中で SQL を使って集計できます。

In [ ]:
con = table.scan().to_duckdb(table_name="trips")
con.sql("""
SELECT CAST(tpep_pickup_datetime AS DATE) AS day,
       payment_type,
       count(*) AS trips,
       round(avg(total_amount), 2) AS avg_total
FROM trips
GROUP BY ALL
ORDER BY day, payment_type
""").df()

## 5. メタデータを見る

`table.inspect` で、Spark や Trino のメタデータテーブル（08）と同じ情報を pyarrow の表として取り出せます。

In [ ]:
table.inspect.snapshots().select(["committed_at", "snapshot_id", "operation"]).to_pandas()

In [ ]:
table.inspect.files().select(["file_path", "record_count", "file_size_in_bytes"]).to_pandas()

## 6. タイムトラベル

スナップショット ID を指定してスキャンすると、その時点の状態を読めます。最初のスナップショット（1 日目だけ）を読んでみます。

In [ ]:
first = table.snapshots()[0].snapshot_id
print("最初の時点:", len(table.scan(snapshot_id=first).to_arrow()), "行")
print("現在:", len(table.scan().to_arrow()), "行")

## 7. Spark や Trino が作ったテーブルを読む

01 で Spark が作った `crud_spark` があれば、PyIceberg からそのまま読めます。

In [ ]:
if catalog.table_exists("handson.crud_spark"):
    print(catalog.load_table("handson.crud_spark").scan().to_pandas())
else:
    print("handson.crud_spark がありません。01 の spark.ipynb を先に実行してください")

## まとめ

- PyIceberg は JVM なしで Iceberg テーブルを作り、読み書きできる
- `row_filter` と `selected_fields` で、必要なファイルと列だけを読める
- pandas や DuckDB と組み合わせて、Python の中で分析できる
- 作ったテーブルは Spark や Trino からもそのまま読める（trino.sql で確認する）